In [5]:
import httpx
import pandas as pd

In [6]:
client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=15.0, write=15.0, pool=15.0),
    limits=httpx.Limits(max_keepalive_connections=0, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers={
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
    },
)

SINA_REFERER_HEADERS = {"Referer": "https://vip.stock.finance.sina.com.cn/mkt/"}


### 新浪行情中心
对应地址 - https://vip.stock.finance.sina.com.cn/mkt/

该行情地址通过`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData?page=1&num=40&sort=symbol&asc=1&node=sh_a&symbol=&_s_r_a=init`（GET）请求行情数据。`page`和`num`分别控制页码和每页数量，`sort`控制排序字段，与`asc`配合控制排序方向（`1`升序、`0`降序）；`node`参数尤其重要，用于筛选不同分类的行情数据。`symbol`通常传空字符串，`_s_r_a`是新浪页面内部参数，有点像request action，比如page代表是翻页触发的请求，init是页面第一次加载等等。

`node`实际的取值范围可参照`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodes`。`hs_a`、`sh_a`、`sz_a`、`hs_bjs`、`cyb`、`kcb`就分别对应沪深 A 股整体、沪 A、深 A、北交所、创业板、科创板，构建 A 股标的池时优先使用这些节点。`hs_s`表示沪深指数集合，`etf_hq_fund` 适合获取场内 ETF 基金行情，是构建 ETF 标的池的首选节点，同时 LOF 基金使用`lof_hq_fund`。

对于接口调用的response数据中，symbol 是带交易所前缀的证券标识（如 bj920000、sh510010），code 是六位证券代码，name 是名称，trade 是最新成交价，pricechange 是相对昨收的涨跌额，changepercent 是涨跌幅（单位为 %），buy 和 sell 分别是当前最优买价和卖价，settlement 是昨收/前一结算价，open、high、low 分别是今开、最高价和最低价，volume 是成交量（股票为股、ETF 为份额），amount 是成交额（元），ticktime 是行情更新时间，per 是市盈率，pb 是市净率；比如安徽凤凰作为股票，这两个字段分别为 18.833 和 1.82，而 180 治理 ETF 返回 0，通常表示不适用或未提供，mktcap 是总市值、nmc 是流通市值，按该接口口径通常以万元计，turnoverratio 是换手率（单位为 %）。

In [7]:
HQ_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData"
HQ_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCount"

In [12]:
HS_A_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'hs_a',
    "symbol": "",
    "_s_r_a": "page",
}

HS_A_COUNT_PARAMS={
    "node": 'hs_a',
}

hs_a_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=HS_A_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())

ha_s_resp = client.request(method="GET", url=HQ_ENDPOINT, params=HS_A_PARAMS, headers=SINA_REFERER_HEADERS)
ha_s_resp_json = ha_s_resp.json()
ha_s_resp_json[0]

{'symbol': 'bj920000',
 'code': '920000',
 'name': '安徽凤凰',
 'trade': '13.560',
 'pricechange': 0.14,
 'changepercent': 1.043,
 'buy': '13.460',
 'sell': '13.560',
 'settlement': '13.420',
 'open': '13.380',
 'high': '13.740',
 'low': '13.000',
 'volume': 656429,
 'amount': 8895246,
 'ticktime': '15:30:01',
 'per': 18.833,
 'pb': 1.82,
 'mktcap': 124318.08,
 'nmc': 78097.3623,
 'turnoverratio': 1.13975}

In [13]:
ETF_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'etf_hq_fund',
    "symbol": "",
    "_s_r_a": "page",
}

ETF_COUNT_PARAMS={
    "node": 'etf_hq_fund',
}

etf_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_counts}")

etf_resp = client.request(method="GET", url=HQ_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_resp_json = etf_resp.json()
etf_resp_json[0]

ETF counts: 1639


{'symbol': 'sh510010',
 'code': '510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': 0.002,
 'changepercent': 0.114,
 'buy': '1.756',
 'sell': '1.762',
 'settlement': '1.754',
 'open': '1.762',
 'high': '1.766',
 'low': '1.749',
 'volume': 41300,
 'amount': 72833,
 'ticktime': '15:00:03',
 'per': 0,
 'pb': 0,
 'mktcap': 23271.2779672,
 'nmc': 22920.08464,
 'turnoverratio': 0.03164}

In [14]:
HQ_SIMPLE_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeDataSimple"
HQ_SIMPLE_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCountSimple"

etf_simple_counts = int(client.request(method="GET", url=HQ_SIMPLE_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_simple_counts}")

etf_simple_resp = client.request(method="GET", url=HQ_SIMPLE_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_simple_resp_json = etf_simple_resp.json()
etf_simple_resp_json[0]

ETF counts: 1640


{'symbol': 'sh510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': '0.002',
 'changepercent': '0.114',
 'buy': '1.756',
 'sell': '1.762',
 'settlement': '1.754',
 'open': '1.762',
 'high': '1.766',
 'low': '1.749',
 'volume': 41300,
 'amount': 72833,
 'code': '510010',
 'ticktime': '15:00:03',
 'state': '00',
 'statetxt': '正常'}